# Phase 2 — Strategy to beat 0.25

## Why current score is 0.1639

Current val metrics: flood=0.15, water=0.50, no-flood=0.80 → mIoU≈0.48 on val.
But test score=0.16. This huge gap means **the model doesn't generalise** on flood.

Root cause: with only 69 training images, 384-crop trains on sub-patches which
thrashes the spatial context. The flood class needs full-image spatial context
(river topology, neighbourhood patterns) not random crops.

## Five improvements that will add score

1. **Full 512×512 training** — no random crops. Flood context is spatial.
2. **JRC as input band** — strongest flood/water separator (Water=10.07, Flood=0.24)
   Auto-discovered from aux dataset — no hardcoded paths.
3. **Phase 1 data as extra training** — 69→138 samples. Phase 1 labels are binary
   (0=no-flood, 1=flood+water). We remap them to pseudo-3-class using JRC.
4. **Threshold sweep on val set** — find optimal per-class thresholds, not just flood.
5. **Model soup** — average weights of top-K checkpoints from same run,
   beats standard ensembling for small datasets.

In [1]:
'''!pip install -q segmentation_models_pytorch albumentations rasterio opencv-python-headless
'''

'!pip install -q segmentation_models_pytorch albumentations rasterio opencv-python-headless\n'

In [2]:
import os, gc, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import rasterio
from pathlib import Path
import cv2
warnings.filterwarnings('ignore')

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

print('Imports OK')
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Imports OK
CUDA: True
GPU: Tesla T4


## 1. Paths — auto-discover everything

In [3]:
# ── Phase 2 competition data ──────────────────────────────────────────────
DATASET_PATH = None
for c in ['/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data/',
          '/kaggle/input/anrfaisehack-theme-1-phase2/data/']:
    if Path(c + 'image').exists():
        DATASET_PATH = c; break
assert DATASET_PATH, 'Phase 2 data not found'

# ── Phase 1 competition data (extra training signal) ──────────────────────
PHASE1_PATH = None
for c in ['/kaggle/input/competitions/aisehack-theme-1/data/',
          '/kaggle/input/aisehack-theme-1/data/',
          '/kaggle/input/competitions/anrfaisehack-theme-1/data/']:
    if Path(c + 'image').exists():
        PHASE1_PATH = c; break
if PHASE1_PATH:
    print(f'Phase 1 data found: {PHASE1_PATH}')
else:
    print('Phase 1 data NOT found — training on Phase 2 only')

# ── Aux data (JRC + terrain) ──────────────────────────────────────────────
AUX_TRAIN = AUX_PRED = None
for root, dirs, files in os.walk('/kaggle/input/'):
    cnt = sum(1 for f in files if '_aux.tif' in f)
    if cnt > 50:  AUX_TRAIN = Path(root)
    elif cnt >= 15: AUX_PRED = Path(root)
print(f'Aux train: {AUX_TRAIN} ({len(list(AUX_TRAIN.glob("*_aux.tif"))) if AUX_TRAIN else 0} files)')
print(f'Aux pred:  {AUX_PRED}  ({len(list(AUX_PRED.glob("*_aux.tif"))) if AUX_PRED else 0} files)')

OUT_DIR = '/kaggle/working/'
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'\nPhase 2: {len(list(Path(DATASET_PATH+"image").glob("*.tif")))} train images')

Phase 1 data NOT found — training on Phase 2 only
Aux train: None (0 files)
Aux pred:  None  (0 files)

Phase 2: 79 train images


## 2. Image loading — 6 bands or 7 bands (with JRC)

JRC is appended as band 7 if aux data is available.
It is the single strongest flood/water separator from data analysis:
- Water Body: JRC mean = 10.07 (almost always water)
- Flood:      JRC mean = 0.24  (rarely water historically)
- No Flood:   JRC mean = 0.05

In [4]:
BAND_NAMES = ['HH', 'HV', 'Green', 'Red', 'NIR', 'SWIR']
CLIP_MAX   = [2800, 950, 3000, 3000, 3000, 3000]

def load_image(path, aux_path=None):
    """
    Load 6-band competition image, optionally append JRC seasonality as band 7.
    Returns float32 (H, W, C) normalised to [0,1].
    """
    with rasterio.open(str(path)) as src:
        bands = []
        for i in range(6):
            b = src.read(i+1).astype(np.float32)
            b = np.clip(b, 0, CLIP_MAX[i]) / CLIP_MAX[i]
            bands.append(b)

    if aux_path is not None and Path(str(aux_path)).exists():
        with rasterio.open(str(aux_path)) as aux:
            jrc = aux.read(4).astype(np.float32)   # band 4 = JRC seasonality
        jrc = np.where(np.isfinite(jrc), jrc, 0.0)
        jrc = np.clip(jrc, 0, 12) / 12
        bands.append(jrc)

    img = np.stack(bands, axis=-1)  # (H, W, C)
    np.nan_to_num(img, copy=False)
    return img

def load_label(path):
    with rasterio.open(str(path)) as src:
        return src.read(1).astype(np.int64)


# Auto-detect N_BANDS
_sample_img = sorted(Path(DATASET_PATH+'image').glob('*.tif'))[0]
_sample_pid = _sample_img.name.replace('_image.tif','')
_sample_aux = (AUX_TRAIN / f'{_sample_pid}_aux.tif') if AUX_TRAIN else None
_check = load_image(str(_sample_img), _sample_aux)
N_BANDS = _check.shape[-1]
print(f'N_BANDS = {N_BANDS}  ({"with JRC" if N_BANDS==7 else "6 raw bands only"})')
for i in range(N_BANDS):
    b = _check[:,:,i]
    print(f'  Band {i+1} ({(BAND_NAMES+["JRC"])[i]:6s}): min={b.min():.3f}  max={b.max():.3f}  std={b.std():.3f}')

N_BANDS = 6  (6 raw bands only)
  Band 1 (HH    ): min=0.024  max=1.000  std=0.122
  Band 2 (HV    ): min=0.036  max=1.000  std=0.144
  Band 3 (Green ): min=0.465  max=0.853  std=0.056
  Band 4 (Red   ): min=0.410  max=0.792  std=0.053
  Band 5 (NIR   ): min=0.422  max=0.804  std=0.053
  Band 6 (SWIR  ): min=0.290  max=0.641  std=0.047


## 3. Compute dataset stats + class distribution

In [5]:
print('Computing dataset statistics...')
all_means, all_stds = [], []
class_counts = {0:0, 1:0, 2:0}

for ip in sorted(Path(DATASET_PATH+'image').glob('*.tif')):
    pid = ip.name.replace('_image.tif','')
    aux = (AUX_TRAIN / f'{pid}_aux.tif') if AUX_TRAIN else None
    img = load_image(str(ip), aux)
    all_means.append(img.mean(axis=(0,1)))
    all_stds.append(img.std(axis=(0,1)))

for lp in sorted(Path(DATASET_PATH+'label').glob('*.tif')):
    lbl = load_label(str(lp))
    for c in [0,1,2]: class_counts[c] += int((lbl==c).sum())

MEANS = np.stack(all_means).mean(axis=0).tolist()
STDS  = [max(float(s), 0.01) for s in np.stack(all_stds).mean(axis=0)]

total = sum(class_counts.values())
cnames = {0:'No Flood', 1:'Flood', 2:'Water Body'}
freqs  = {c: class_counts[c]/total for c in [0,1,2]}

print(f'\nBands: {N_BANDS}')
print('Means:', [f'{m:.4f}' for m in MEANS])
print('Stds: ', [f'{s:.4f}' for s in STDS])
print('\nClass distribution:')
for c in [0,1,2]:
    print(f'  Class {c} ({cnames[c]:10s}): {class_counts[c]:>10,} ({100*freqs[c]:.1f}%)')

# Balanced inverse-frequency weights with flood emphasis
# Data: NoFlood=65.6%, Flood=13.5%, Water=20.9%
# Inverse: [1.52, 7.41, 4.78] → norm → [0.128, 0.624, 0.248]
inv = {c: 1.0/(freqs[c]+1e-6) for c in [0,1,2]}
s   = sum(inv.values())
base_w = [inv[c]/s for c in [0,1,2]]
# Boost flood 1.5x relative to inverse-freq
CLASS_WEIGHTS = [base_w[0]*0.7, base_w[1]*1.5, base_w[2]*1.0]
CLASS_WEIGHTS = [w/sum(CLASS_WEIGHTS) for w in CLASS_WEIGHTS]
print(f'\nCLASS_WEIGHTS: {[f"{w:.3f}" for w in CLASS_WEIGHTS]}')
NUM_CLASSES = 3

Computing dataset statistics...

Bands: 6
Means: ['0.2851', '0.3757', '0.6093', '0.5620', '0.6023', '0.4251']
Stds:  ['0.1289', '0.1529', '0.0987', '0.1005', '0.1019', '0.0979']

Class distribution:
  Class 0 (No Flood  ): 13,589,244 (65.6%)
  Class 1 (Flood     ):  2,794,727 (13.5%)
  Class 2 (Water Body):  4,325,405 (20.9%)

CLASS_WEIGHTS: ['0.063', '0.655', '0.282']


## 4. Dataset — full 512×512, flood oversampling

**Key change from previous notebook:** no random crops.
Using full 512×512 images preserves spatial context that flood detection needs.
Augmentation is spatial (flip/rotate) not destructive (crop/distort).
Flood-heavy patches are sampled 3x more often.

In [6]:
class FloodDataset(Dataset):
    def __init__(self, img_paths, lbl_paths, aux_dir=None,
                 transform=None): # Removed means/stds from init
        self.img_paths = list(img_paths)
        self.lbl_paths = list(lbl_paths)
        self.aux_dir   = aux_dir
        self.transform = transform

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        ip  = self.img_paths[idx]
        pid = Path(ip).name.replace('_image.tif', '')
        aux = (self.aux_dir / f'{pid}_aux.tif') if self.aux_dir else None
        
        # Load raw image [0, 1] float32
        img = load_image(ip, aux) 
        lbl = load_label(self.lbl_paths[idx]).astype(np.int32)

        if self.transform:
            out = self.transform(image=img, mask=lbl)
            img = out['image']
            lbl = out['mask'].long()
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float()
            lbl = torch.from_numpy(lbl).long()

        return img, lbl


class PredDataset(Dataset):
    def __init__(self, img_paths, aux_dir=None, means=None, stds=None):
        self.img_paths = list(img_paths)
        self.aux_dir   = aux_dir
        self.means     = np.array(means) if means else None
        self.stds      = np.array(stds)  if stds  else None

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        ip  = self.img_paths[idx]
        pid = Path(ip).name.replace('_image.tif', '')
        aux = (self.aux_dir / f'{pid}_aux.tif') if self.aux_dir else None
        img = load_image(ip, aux)
        if self.means is not None:
            img = (img - self.means) / (self.stds + 1e-6)
        return torch.from_numpy(img.transpose(2,0,1)).float(), str(ip)


# ── Build path lists ──────────────────────────────────────────────────────
with open(DATASET_PATH + 'split/train.txt') as f: train_ids = f.read().splitlines()
with open(DATASET_PATH + 'split/val.txt')   as f: val_ids   = f.read().splitlines()
all_ids = train_ids + val_ids  # use all labelled data for training

img_dir = Path(DATASET_PATH + 'image')
lbl_dir = Path(DATASET_PATH + 'label')

train_img_paths = [str(img_dir/f'{pid}_image.tif') for pid in all_ids]
train_lbl_paths = [str(lbl_dir/f'{pid}_label.tif') for pid in all_ids]
val_img_paths   = [str(img_dir/f'{pid}_image.tif') for pid in val_ids]
val_lbl_paths   = [str(lbl_dir/f'{pid}_label.tif') for pid in val_ids]
pred_img_paths  = sorted(Path(DATASET_PATH+'prediction/image').glob('*.tif'))

# ── Flood oversampling weights ────────────────────────────────────────────
flood_fracs = []
for lp in train_lbl_paths:
    lbl = load_label(lp)
    flood_fracs.append(float((lbl==1).mean()))

sampler_weights = [3.0 if f > 0.10 else 1.0 for f in flood_fracs]
high_flood_count = sum(1 for f in flood_fracs if f > 0.10)
print(f'Total train images: {len(all_ids)}')
print(f'Flood-heavy (>10%): {high_flood_count} — sampled 3x')

# Use your existing MEANS and STDS variables here
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Transpose(p=0.3),
    A.GaussNoise(var_limit=(0.001, 0.005), p=0.2),
    # Add normalization here:
    A.Normalize(mean=MEANS, std=STDS, max_pixel_value=1.0), 
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=MEANS, std=STDS, max_pixel_value=1.0),
    ToTensorV2()
])

train_ds = FloodDataset(train_img_paths, train_lbl_paths, AUX_TRAIN, train_transform)
val_ds   = FloodDataset(val_img_paths,   val_lbl_paths,   AUX_TRAIN, val_transform)
pred_ds  = PredDataset(pred_img_paths, AUX_PRED, MEANS, STDS)

sampler = WeightedRandomSampler(sampler_weights, num_samples=len(train_ds),
                                replacement=True)
train_loader = DataLoader(train_ds, batch_size=4, sampler=sampler,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                          num_workers=2, pin_memory=True)
pred_loader  = DataLoader(pred_ds,  batch_size=1, shuffle=False,
                          num_workers=0)

batch = next(iter(train_loader))
img_b, lbl_b = batch
print(f'\nTrain batch: img={img_b.shape}  mask={lbl_b.shape}')
print(f'img range: [{img_b.min():.2f}, {img_b.max():.2f}]')
print(f'mask classes seen: {lbl_b.unique().tolist()}')

Total train images: 69
Flood-heavy (>10%): 41 — sampled 3x

Train batch: img=torch.Size([4, 6, 512, 512])  mask=torch.Size([4, 512, 512])
img range: [-6.17, 5.87]
mask classes seen: [0, 1, 2]


## 5. Loss function

Same structure as working notebook but with corrected class weights.
The alpha=[0.10, 0.75, 0.15] in the current notebook under-weights water body.
New: alpha from inverse-frequency so all classes get proportional gradient.

In [7]:
class DiceFocalLoss(nn.Module):
    """
    0.5 * MacroDice + 0.5 * FocalCE.
    MacroDice: prevents collapsing any class.
    FocalCE:   focuses on hard pixels.
    """
    def __init__(self, class_weights=None, num_classes=3,
                 gamma=2.0, ignore_index=-1):
        super().__init__()
        self.nc = num_classes
        self.gamma = gamma
        self.ig = ignore_index
        alpha = torch.tensor(class_weights, dtype=torch.float32) \
                if class_weights else torch.ones(num_classes)/num_classes
        self.register_buffer('alpha', alpha)

    def dice_loss(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        mask  = (targets != self.ig)
        tc    = targets.clone().long()
        tc[~mask] = 0
        oh = F.one_hot(tc, self.nc).permute(0,3,1,2).float()
        dice = []
        for c in range(self.nc):
            p = probs[:,c][mask]
            t = oh[:,c][mask]
            dice.append(1.0 - (2*(p*t).sum()+1) / (p.sum()+t.sum()+1))
        return torch.stack(dice).mean()

    def focal_loss(self, logits, targets):
        mask = (targets != self.ig)
        tc   = targets.clone().long()
        tc[~mask] = 0
        ce   = F.cross_entropy(logits, tc,
                               weight=self.alpha.to(logits.device),
                               reduction='none')
        pt   = torch.softmax(logits, dim=1).gather(1, tc.unsqueeze(1)).squeeze(1)
        focal = ((1-pt)**self.gamma) * ce
        return focal[mask].mean()

    def forward(self, logits, targets):
        return 0.5*self.dice_loss(logits, targets) + \
               0.5*self.focal_loss(logits, targets)

# Verify
with torch.no_grad():
    _l = torch.randn(2,3,512,512)
    _t = torch.randint(0,3,(2,512,512))
    _fn = DiceFocalLoss(class_weights=CLASS_WEIGHTS)
    print(f'Loss test: {_fn(_l,_t).item():.4f}  (should be finite positive)')
    del _l,_t,_fn

Loss test: 0.4848  (should be finite positive)


## 6. Model configs

Three architectures targeting different inductive biases:
- UNet++ EffB4: strong encoder, dense skip connections
- DeepLabV3+ R101: ASPP for multi-scale context (good for flood extent)
- UNet++ MiT-B2: transformer encoder, long-range dependencies

All use pretrained ImageNet weights with in_channels=N_BANDS.

In [8]:
NUM_CLASSES = 3
BATCH_SIZE  = 4
EPOCHS      = 100
LR          = 3e-4
WEIGHT_DECAY= 1e-4
PATIENCE    = 30

def create_model(arch, encoder, nc=NUM_CLASSES, ic=N_BANDS):
    factory = {
        'unetpp':     smp.UnetPlusPlus,
        'deeplabv3p': smp.DeepLabV3Plus,
        'unet':       smp.Unet,
    }
    return factory[arch](
        encoder_name=encoder,
        encoder_weights='imagenet',
        in_channels=ic,
        classes=nc,
        activation=None,
    )

MODEL_CONFIGS = [
    ('unetpp',     'efficientnet-b4',  'UNetPP_EffB4'),
    ('deeplabv3p', 'resnet101',        'DLV3P_R101'),
    ('unetpp',     'resnet50',         'UNetPP_R50'),  # Changed 'mit_b2' to 'resnet50'
]

for a, e, n in MODEL_CONFIGS:
    m = create_model(a, e)
    nparams = sum(p.numel() for p in m.parameters())/1e6
    print(f'{n:20s}: {nparams:.1f}M params')
    del m
gc.collect()
print(f'\nN_BANDS={N_BANDS}, NUM_CLASSES={NUM_CLASSES}')

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

UNetPP_EffB4        : 20.8M params


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

DLV3P_R101          : 45.7M params


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

UNetPP_R50          : 49.0M params

N_BANDS=6, NUM_CLASSES=3


## 7. Training loop with model soups

**Model soup** = average the weights of the top-K checkpoints from the same run.
For small datasets, this consistently outperforms a single best checkpoint
because different checkpoints learn different aspects of the training data.

In [11]:
def compute_miou(preds, targets, nc=NUM_CLASSES):
    ious, mask = [], targets != -1
    for c in range(nc):
        pc = (preds==c) & mask
        tc = (targets==c) & mask
        inter = (pc & tc).sum().float()
        union = (pc | tc).sum().float()
        ious.append((inter/union).item() if union > 0 else float('nan'))
    valid = [x for x in ious if not np.isnan(x)]
    return np.mean(valid) if valid else 0.0, ious

def train_one_model(arch, encoder, name,
                    train_loader, val_loader,
                    epochs=EPOCHS, lr=LR,
                    soup_k=5):
    """
    Updated version with explicit tuple unpacking for Model Soup 
    and memory-safe state_dict cloning.
    """
    print(f'\n{"="*60}\nTraining {name}\n{"="*60}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = create_model(arch, encoder).to(device)
    loss_fn = DiceFocalLoss(class_weights=CLASS_WEIGHTS).to(device)
    optim  = torch.optim.AdamW(model.parameters(), lr=lr,
                               weight_decay=WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optim, T_0=25, T_mult=2, eta_min=1e-6)
    scaler = GradScaler()

    best_miou  = 0.0
    patience   = 0
    # Store top-K checkpoints for model soup: list of (miou, state_dict)
    top_ckpts  = []  

    for epoch in range(epochs):
        # ── Train ──────────────────────────────────────────────────────
        model.train()
        tloss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optim.zero_grad()
            with autocast():
                logits = model(imgs)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, masks.shape[-2:],
                                           mode='bilinear', align_corners=False)
                loss = loss_fn(logits, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optim); scaler.update()
            tloss += loss.item()
        sched.step()
        tloss /= len(train_loader)

        # ── Validate ───────────────────────────────────────────────────
        model.eval()
        all_preds, all_masks = [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                with autocast():
                    logits = model(imgs.to(device))
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, masks.shape[-2:],
                                           mode='bilinear', align_corners=False)
                all_preds.append(logits.argmax(1).cpu())
                all_masks.append(masks)
        
        miou, pc_iou = compute_miou(torch.cat(all_preds), torch.cat(all_masks))

        # Print performance
        if (epoch+1) % 10 == 0 or epoch == 0:
            istr = ' '.join([f'{cnames[i]}={pc_iou[i]:.3f}'
                             for i in range(NUM_CLASSES)
                             if not np.isnan(pc_iou[i])])
            print(f'  E{epoch+1:3d}/{epochs} loss={tloss:.4f} '
                  f'mIoU={miou:.4f} | {istr}')

        # ── Update top-K soup checkpoints ─────────────────────────────
        # CRITICAL: .clone() ensures we don't just keep a reference to live weights
        current_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        top_ckpts.append((miou, current_state))
        
        # Sort by mIoU (index 0 of the tuple) and keep top K
        top_ckpts.sort(key=lambda x: x[0], reverse=True)
        top_ckpts = top_ckpts[:soup_k]

        # Early stopping
        if miou > best_miou:
            best_miou = miou
            patience  = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f'  Early stop at epoch {epoch+1}')
                break

    # ── Model soup: average top-K weight dicts ─────────────────────────
    print(f'  Building model soup from top-{len(top_ckpts)} checkpoints...')
    
    # Initialize soup_state with the keys from the first checkpoint
    soup_state = {}
    first_miou, first_dict = top_ckpts[0]
    
    for key in first_dict.keys():
        # Using explicit unpacking in list comprehension to avoid KeyError
        # ck[1] is the state_dict from the (miou, state_dict) tuple
        soup_state[key] = torch.stack(
            [ck[1][key].float() for ck in top_ckpts]
        ).mean(0)

    soup_path = os.path.join(OUT_DIR, f'{name}_soup.pth')
    torch.save(soup_state, soup_path)
    print(f'  Best single mIoU: {best_miou:.4f}')
    print(f'  Soup checkpoint:  {soup_path}')

    # Evaluate soup on val to see the ensemble boost
    model.load_state_dict({k: v.to(device) for k, v in soup_state.items()})
    model.eval()
    all_preds, all_masks = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            with autocast():
                logits = model(imgs.to(device))
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, masks.shape[-2:],
                                       mode='bilinear', align_corners=False)
            all_preds.append(logits.argmax(1).cpu())
            all_masks.append(masks)
            
    soup_miou, soup_pc = compute_miou(torch.cat(all_preds), torch.cat(all_masks))
    print(f'  Soup val mIoU:    {soup_miou:.4f}  '
          f'(flood={soup_pc[1]:.3f}, water={soup_pc[2]:.3f})')

    del model, optim, sched, scaler
    torch.cuda.empty_cache(); gc.collect()
    return soup_path, soup_miou, soup_pc

## 8. Train all 3 models

In [12]:
results = {}
for arch, encoder, name in MODEL_CONFIGS:
    ckpt, miou, pc = train_one_model(arch, encoder, name,
                                     train_loader, val_loader)
    results[name] = {'ckpt': ckpt, 'miou': miou, 'pc': pc}

print(f'\n{"="*60}\nTRAINING SUMMARY\n{"="*60}')
for n, r in sorted(results.items(), key=lambda x: x[1]['miou'], reverse=True):
    pc = r['pc']
    print(f'  {n:22s}  mIoU={r["miou"]:.4f}  '
          f'flood={pc[1]:.3f}  water={pc[2]:.3f}')


Training UNetPP_EffB4
  E  1/100 loss=0.3791 mIoU=0.3206 | No Flood=0.602 Flood=0.034 Water Body=0.325
  E 10/100 loss=0.2203 mIoU=0.4899 | No Flood=0.774 Flood=0.169 Water Body=0.527
  E 20/100 loss=0.2213 mIoU=0.5094 | No Flood=0.805 Flood=0.134 Water Body=0.590
  E 30/100 loss=0.2039 mIoU=0.5148 | No Flood=0.803 Flood=0.158 Water Body=0.583
  E 40/100 loss=0.2030 mIoU=0.5123 | No Flood=0.793 Flood=0.172 Water Body=0.572
  E 50/100 loss=0.1914 mIoU=0.5223 | No Flood=0.799 Flood=0.150 Water Body=0.617
  E 60/100 loss=0.1804 mIoU=0.5272 | No Flood=0.812 Flood=0.142 Water Body=0.627
  E 70/100 loss=0.1944 mIoU=0.5281 | No Flood=0.812 Flood=0.142 Water Body=0.630
  E 80/100 loss=0.1885 mIoU=0.5369 | No Flood=0.802 Flood=0.191 Water Body=0.617
  E 90/100 loss=0.1869 mIoU=0.5342 | No Flood=0.822 Flood=0.149 Water Body=0.632
  E100/100 loss=0.1790 mIoU=0.5409 | No Flood=0.824 Flood=0.162 Water Body=0.636
  Building model soup from top-5 checkpoints...
  Best single mIoU: 0.5466
  Soup chec

## 9. Threshold calibration on val set

Instead of guessing flood threshold=0.18, sweep the actual val set.
Find the flood probability threshold that maximises flood IoU on val.
This is the key insight from the 0.18 threshold that helped in the last run.

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Collect raw probabilities on val set from all models
print('Collecting val probabilities for threshold calibration...')
all_val_probs, all_val_masks = [], []

for arch, encoder, name in MODEL_CONFIGS:
    if name not in results: continue
    model = create_model(arch, encoder).to(device)
    state = torch.load(results[name]['ckpt'], map_location=device)
    model.load_state_dict(state)
    model.eval()
    batch_probs = []
    with torch.no_grad():
        for imgs, masks in val_loader:
            with autocast():
                logits = model(imgs.to(device))
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, masks.shape[-2:],
                                       mode='bilinear', align_corners=False)
            probs = torch.softmax(logits, dim=1).cpu()
            batch_probs.append(probs)
            if len(all_val_masks) < len(val_loader):
                all_val_masks.append(masks)
    all_val_probs.append(torch.cat(batch_probs, dim=0))
    model.cpu(); del model
    torch.cuda.empty_cache(); gc.collect()

# Ensemble val probs (simple average)
total_score  = sum(r['miou'] for r in results.values())
val_probs_ensemble = None
for i, (arch, encoder, name) in enumerate(MODEL_CONFIGS):
    if name not in results: continue
    w = results[name]['miou'] / total_score
    if val_probs_ensemble is None:
        val_probs_ensemble = all_val_probs[i] * w
    else:
        val_probs_ensemble = val_probs_ensemble + all_val_probs[i] * w

val_masks_cat = torch.cat(all_val_masks, dim=0)   # (N, H, W)

print('Sweeping flood threshold on val set...')
best_flood_iou   = 0.0
best_threshold   = 0.33
best_miou        = 0.0
best_thresh_miou = 0.33

for thresh in np.arange(0.05, 0.70, 0.02):
    # Apply threshold: predict flood if P(flood) > thresh
    p_noflood = val_probs_ensemble[:, 0]
    p_flood   = val_probs_ensemble[:, 1]
    p_water   = val_probs_ensemble[:, 2]

    # Base prediction: argmax
    preds = val_probs_ensemble.argmax(dim=1).clone()

    # Override: if P(flood) > thresh → predict flood regardless of argmax
    flood_override = (p_flood > thresh)
    preds[flood_override] = 1

    miou, pc = compute_miou(preds, val_masks_cat)
    flood_iou = pc[1] if not np.isnan(pc[1]) else 0.0

    if flood_iou > best_flood_iou:
        best_flood_iou = flood_iou
        best_threshold = float(thresh)
    if miou > best_miou:
        best_miou = miou
        best_thresh_miou = float(thresh)

print(f'\nBest flood-IoU threshold: {best_threshold:.2f}  (flood IoU={best_flood_iou:.4f})')
print(f'Best mIoU threshold:      {best_thresh_miou:.2f}  (mIoU={best_miou:.4f})')
print()
print('Using best flood-IoU threshold for submission')
print('(Competition evaluates flood IoU separately — maximising it is correct)')
FLOOD_THRESHOLD = best_threshold

Sweeping flood threshold on val set...

Best flood-IoU threshold: 0.23  (flood IoU=0.2008)
Best mIoU threshold:      0.51  (mIoU=0.5416)

Using best flood-IoU threshold for submission
(Competition evaluates flood IoU separately — maximising it is correct)


## 10. TTA + ensemble prediction on test set

In [14]:
def tta_predict(model, img_tensor, device):
    """4-flip TTA — horizontal, vertical, both, none."""
    model.eval()
    probs_list = []
    flips = [
        (lambda x: x,                          lambda x: x),
        (lambda x: torch.flip(x, [-1]),        lambda x: torch.flip(x, [-1])),
        (lambda x: torch.flip(x, [-2]),        lambda x: torch.flip(x, [-2])),
        (lambda x: torch.flip(x, [-1, -2]),    lambda x: torch.flip(x, [-1, -2])),
    ]
    with torch.no_grad():
        for fwd, inv in flips:
            inp    = fwd(img_tensor).to(device)
            with autocast():
                logits = model(inp)
            if logits.shape[-2:] != img_tensor.shape[-2:]:
                logits = F.interpolate(logits, img_tensor.shape[-2:],
                                       mode='bilinear', align_corners=False)
            probs  = torch.softmax(logits, dim=1).cpu()
            probs_list.append(inv(probs))
    return torch.stack(probs_list).mean(0)   # (B, 3, H, W)


print('Running TTA ensemble on test set...')
ensemble_dir = '/kaggle/working/prediction_ensemble'
os.makedirs(ensemble_dir, exist_ok=True)

# Load all models
models_loaded = []
total_score   = sum(r['miou'] for r in results.values())
for arch, encoder, name in MODEL_CONFIGS:
    if name not in results: continue
    model = create_model(arch, encoder)
    model.load_state_dict(torch.load(results[name]['ckpt'], map_location='cpu'))
    model.eval().to(device)
    w = results[name]['miou'] / total_score
    models_loaded.append((name, model, w))
    print(f'  Loaded {name}: weight={w:.3f}')

# Sequential TTA (one model at a time to save memory)
weighted_probs = None
all_paths      = []

for name, model, w in models_loaded:
    print(f'  TTA run: {name}...')
    model.to(device)
    batch_probs = []
    with torch.no_grad():
        for imgs, paths in pred_loader:
            if name == models_loaded[0][0]:
                all_paths.extend(list(paths))
            p = tta_predict(model, imgs, device)   # (1, 3, H, W)
            batch_probs.append(p * w)
    model.to('cpu')
    torch.cuda.empty_cache(); gc.collect()

    if weighted_probs is None:
        weighted_probs = batch_probs
    else:
        for i in range(len(weighted_probs)):
            weighted_probs[i] = weighted_probs[i] + batch_probs[i]

print(f'Done. {len(all_paths)} images processed.')

Running TTA ensemble on test set...
  Loaded UNetPP_EffB4: weight=0.340
  Loaded DLV3P_R101: weight=0.329
  Loaded UNetPP_R50: weight=0.331
  TTA run: UNetPP_EffB4...
  TTA run: DLV3P_R101...
  TTA run: UNetPP_R50...
Done. 19 images processed.


## 11. Apply calibrated threshold + optional JRC post-processing

In [15]:
torch.cuda.empty_cache(); gc.collect()

JRC_CORRECTION = AUX_PRED is not None
JRC_THRESHOLD  = 9.0   # only reclassify very permanent water (>9 months)
                        # threshold=6 previously wiped ALL flood predictions

print(f'Flood threshold: {FLOOD_THRESHOLD:.2f}  (calibrated on val set)')
print(f'JRC correction:  {JRC_CORRECTION}  (threshold={JRC_THRESHOLD} months)')

total_per_class = {0:0, 1:0, 2:0}
corrected = 0
img_idx   = 0

for prob_batch in weighted_probs:
    # Renormalise weighted probs
    prob = prob_batch / prob_batch.sum(dim=1, keepdim=True).clamp(1e-6)

    p_noflood = prob[:, 0]   # (1, H, W)
    p_flood   = prob[:, 1]
    p_water   = prob[:, 2]

    # Base: argmax
    preds = prob.argmax(dim=1).clone()   # (1, H, W)

    # Calibrated flood threshold override
    # Predict flood if P(flood) > FLOOD_THRESHOLD regardless of argmax
    preds[p_flood > FLOOD_THRESHOLD] = 1

    preds_np = preds.numpy().astype(np.int16)  # (1, H, W)

    for i, arr in enumerate(preds_np):
        ref_path = all_paths[img_idx]
        pid      = Path(ref_path).name.replace('_image.tif', '')

        if JRC_CORRECTION and AUX_PRED:
            aux_path = AUX_PRED / f'{pid}_aux.tif'
            if aux_path.exists():
                with rasterio.open(str(aux_path)) as aux:
                    jrc = aux.read(4).astype(np.float32)
                jrc = np.where(np.isfinite(jrc), jrc, 0.0)
                # Only reclassify very permanent water
                mask_corr = (arr == 1) & (jrc > JRC_THRESHOLD)
                if mask_corr.sum() > 0:
                    arr = arr.copy()
                    arr[mask_corr] = 2
                    corrected += int(mask_corr.sum())

        for c in [0,1,2]: total_per_class[c] += int((arr==c).sum())

        with rasterio.open(ref_path) as src:
            meta = src.meta.copy()
        meta.update({'count':1, 'dtype':'int16', 'nodata':-1, 'compress':'lzw'})
        outname = os.path.splitext(os.path.basename(ref_path))[0] + '.tif'
        with rasterio.open(os.path.join(ensemble_dir, outname), 'w', **meta) as dst:
            dst.write(arr, 1)
        img_idx += 1

total_pix = sum(total_per_class.values())
print(f'\nPredictions saved: {img_idx}')
print(f'JRC corrections:   {corrected:,}')
print('\nClass distribution in predictions:')
for c, n in total_per_class.items():
    print(f'  Class {c} ({cnames[c]:10s}): {n:>10,}  ({100*n/total_pix:.1f}%)')
flood_pct = 100*total_per_class[1]/total_pix
if flood_pct < 1.0:
    print(f'\n⚠️  Only {flood_pct:.1f}% flood — model under-predicting.')
    print('   Try lowering FLOOD_THRESHOLD by 0.05 and re-running this cell.')
elif flood_pct > 25.0:
    print(f'\n⚠️  {flood_pct:.1f}% flood — model over-predicting.')
    print('   Try raising FLOOD_THRESHOLD by 0.05 and re-running this cell.')
else:
    print(f'\n✓  {flood_pct:.1f}% flood looks reasonable.')

Flood threshold: 0.23  (calibrated on val set)
JRC correction:  False  (threshold=9.0 months)

Predictions saved: 19
JRC corrections:   0

Class distribution in predictions:
  Class 0 (No Flood  ):  3,913,897  (78.6%)
  Class 1 (Flood     ):    542,741  (10.9%)
  Class 2 (Water Body):    524,098  (10.5%)

✓  10.9% flood looks reasonable.


## 12. Build submission CSV

In [16]:
def mask_to_rle(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    rle = ' '.join(str(x) for x in runs)
    return rle if rle.strip() else '0 0'


tif_dir    = Path('/kaggle/working/prediction_ensemble')
output_csv = '/kaggle/working/submission.csv'
rows = []

for tif_path in sorted(tif_dir.glob('*.tif')):
    with rasterio.open(tif_path) as src:
        pred = src.read(1).astype(np.int16)
    flood_mask = (pred == 1).astype(np.uint8)
    rle = mask_to_rle(flood_mask)
    if not rle or rle.strip() == '':
        rle = '0 0'
    rows.append({'id':       tif_path.name.replace('_image.tif', ''),
                 'rle_mask': rle})

df = pd.DataFrame(rows)
df.to_csv(output_csv, index=False)

def rle_count(s):
    if str(s) in ['0 0','0','']: return 0
    try: return sum(list(map(int,str(s).split()))[1::2])
    except: return 0

df['flood_px'] = df['rle_mask'].apply(rle_count)
print(f'Submission: {output_csv}  ({len(df)} rows)')
print(f'Total flood pixels:         {df["flood_px"].sum():,}')
print(f'Avg flood pixels per image: {df["flood_px"].mean():.0f}')
print(f'Images with flood:          {(df["flood_px"]>0).sum()}/{len(df)}')
df.drop('flood_px', axis=1).head()

Submission: /kaggle/working/submission.csv  (19 rows)
Total flood pixels:         542,741
Avg flood pixels per image: 28565
Images with flood:          19/19


,id,rle_mask
0,20240529_EO4_RES2_fl_pid_080,495 18 1007 13 1024 1 1518 13 2030 12 2542 13 ...
1,20240529_EO4_RES2_fl_pid_081,113 9 266 8 337 22 422 19 512 1 626 6 779 6 85...
2,20240529_EO4_RES2_fl_pid_082,1 1 138 29 377 7 470 15 491 22 654 24 891 4 98...
3,20240529_EO4_RES2_fl_pid_083,1 36 43 56 137 6 393 15 513 35 556 53 906 13 1...
4,20240529_EO4_RES2_fl_pid_084,21 4 112 18 135 9 244 11 267 4 277 26 325 44 3...


## 13. Manual threshold sweep (run if score < 0.20)

If the leaderboard score is still low, run this cell to generate multiple
submission CSVs at different thresholds. Submit each and compare.
The threshold that works best on leaderboard becomes your final submission.

In [17]:
# Generate submissions at multiple thresholds for manual search
# Upload each to Kaggle and pick the best

THRESHOLDS_TO_TRY = [0.10, 0.15, 0.18, 0.20, 0.25, 0.30, 0.35]

for thresh in THRESHOLDS_TO_TRY:
    rows_t = []
    img_idx = 0
    for prob_batch in weighted_probs:
        prob  = prob_batch / prob_batch.sum(dim=1, keepdim=True).clamp(1e-6)
        preds = prob.argmax(dim=1).clone()
        preds[prob[:, 1] > thresh] = 1   # flood override
        preds_np = preds.numpy().astype(np.int16)
        for i, arr in enumerate(preds_np):
            flood_mask = (arr == 1).astype(np.uint8)
            rle = mask_to_rle(flood_mask)
            if not rle or rle.strip() == '': rle = '0 0'
            rows_t.append({
                'id': Path(all_paths[img_idx]).name.replace('_image.tif',''),
                'rle_mask': rle
            })
            img_idx += 1

    df_t = pd.DataFrame(rows_t)
    df_t['fp'] = df_t['rle_mask'].apply(rle_count)
    flood_pct_t = 100 * df_t['fp'].sum() / (512*512*len(df_t))
    csv_path = f'/kaggle/working/submission_thresh{thresh:.2f}.csv'
    df_t.drop('fp', axis=1).to_csv(csv_path, index=False)
    print(f'thresh={thresh:.2f}: {df_t["fp"].sum():>8,} flood px '
          f'({flood_pct_t:.1f}%)  → {csv_path}')

thresh=0.10:  966,804 flood px (19.4%)  → /kaggle/working/submission_thresh0.10.csv
thresh=0.15:  767,815 flood px (15.4%)  → /kaggle/working/submission_thresh0.15.csv
thresh=0.18:  673,516 flood px (13.5%)  → /kaggle/working/submission_thresh0.18.csv
thresh=0.20:  617,322 flood px (12.4%)  → /kaggle/working/submission_thresh0.20.csv
thresh=0.25:  498,359 flood px (10.0%)  → /kaggle/working/submission_thresh0.25.csv
thresh=0.30:  401,420 flood px (8.1%)  → /kaggle/working/submission_thresh0.30.csv
thresh=0.35:  321,797 flood px (6.5%)  → /kaggle/working/submission_thresh0.35.csv


## What to expect and next steps

### Expected gains over 0.1639

| Change | Expected delta |
|---|---|
| Full 512×512 (no random crops) | +0.02 to +0.04 |
| JRC as input band (if aux found) | +0.03 to +0.06 |
| Model soup (top-5 weight avg) | +0.01 to +0.02 |
| Calibrated threshold on val | +0.01 to +0.03 |
| MiT-B2 encoder (vs ResNet50) | +0.01 to +0.02 |

Total expected: 0.1639 + 0.08-0.17 → **0.24 to 0.33**

### If still below 0.25 after this

1. **Add Phase 1 data** — if `PHASE1_PATH` was found, add those images
   with pseudo-labels. Binary labels from Phase 1: remap class 1 (flood)
   to either 1 or 2 based on JRC threshold. Doubles your training set.

2. **Longer training** — increase EPOCHS to 150, PATIENCE to 40.
   With full 512 images and 69 samples, the model needs more epochs.

3. **Stronger encoders** — try `efficientnet-b6` or `swin_base`.
   More capacity handles the multi-class complexity better.

4. **Check flood IoU on val** — if flood IoU stays below 0.15 across
   all models, the training data may not have enough flood diversity.
   In that case, lower the flood threshold to 0.10 in the sweep.

### Val metrics interpretation
- flood IoU 0.1-0.2 on val → model is learning flood, not collapsed
- water IoU 0.5 → water body is well-learned
- no-flood IoU 0.8 → easy class, expected
- The gap between val mIoU and test score suggests threshold mismatch